# LoCED — MagicNorm + mHC architecture validation (Kaggle)

This notebook produces the evidence for the opt-in architecture work on branch
`architecture-mhc-magicnorm`:

- **MagicNorm** — module-level RMSNorm at every visit exit.
- **mHC** — four parallel residual streams, with `H_res` projected onto the
  Birkhoff polytope by Sinkhorn, softmax `H_pre` / `H_post`, and optional
  per-token dynamic coefficients.

It is intentionally a **validation** notebook, not a long training run.

### What it checks

1. The full reference smoke suite (**31 tests**).
2. The MaxText v0.2.5 integration suite with **real MaxText utilities**.
3. Reference ↔ NNX parity for four configurations:
   - baseline;
   - MagicNorm only;
   - static mHC (4 streams) + MagicNorm;
   - dynamic mHC + MagicNorm.
   Parity covers parameters, loss/logits, gradients, prefill store/logits, and
   decode logits, with `reuse_p=0.5`.
4. MaxText Muon one-step training with mHC + MagicNorm enabled.
5. A small throughput/memory comparison between baseline and mHC + MagicNorm.

### Kaggle checklist

1. **Accelerator → TPU v5e-8** (or GPU; edit `ACCELERATOR` below).
2. **Internet → ON**.
3. The branch must be available on GitHub. The notebook clones
   `https://github.com/offx-zinth/loced.git` at
   `architecture-mhc-magicnorm` by default. If the branch is not pushed yet,
   push it first or set `LOCED_REPO_REF` to a commit/branch that contains the
   MagicNorm/mHC commits.
4. Run all cells in order. The first install takes roughly 5–15 minutes.

No Hugging Face token is needed: all training data here is synthetic.

## 0. Configuration — repository, paths, accelerator

The notebook keeps MaxText and its pinned JAX stack in an isolated Python 3.12
environment. Set `ACCELERATOR="gpu"` if you are validating the GPU path instead
of TPU.

In [ ]:
# ============================================================================
# LoCED MagicNorm + mHC - Kaggle validation
# ============================================================================
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

# Applied only when the architecture branch has not been pushed to GitHub yet.
ARCH_PATCH = r'''diff --git a/block.py b/block.py
index 82dfeae..84cbe2d 100644
--- a/block.py
+++ b/block.py
@@ -29,17 +29,35 @@ def rmsnorm(cfg: LoCEDConfig, x, w):
     return (x * w.astype(x.dtype)).astype(x.dtype)
 
 
+def sinkhorn(logits, iters: int, eps: float = 1e-6):
+    """Project a matrix onto the Birkhoff polytope (doubly stochastic).
+
+    This is the mHC manifold constraint: every row and column sums to one, so
+    the residual mixing matrix has spectral norm <= 1 and cannot amplify the
+    residual stream. Works on the last two axes, so dynamic per-token
+    coefficients are supported too.
+    """
+    h = jax.nn.softmax(logits, axis=-1)
+    for _ in range(iters):
+        h = h / (h.sum(axis=-2, keepdims=True) + eps)
+        h = h / (h.sum(axis=-1, keepdims=True) + eps)
+    return h
+
+
 def init_layer(cfg: LoCEDConfig, key: jax.Array) -> dict:
     """ONE physical layer's own parameters: pre-norms + MoE FFN.
 
     (Attention compute is merged in by `init_model` via `init_layer_attn`.)
     """
     d = cfg.hidden_size
-    return dict(
+    out = dict(
         norm_a=jnp.ones((d,), jnp.float32),
         norm_m=jnp.ones((d,), jnp.float32),
         **init_moe(cfg, key),
     )
+    if cfg.magic_norm:
+        out["norm_magic"] = jnp.ones((d,), jnp.float32)
+    return out
 
 
 def init_loop(cfg: LoCEDConfig, key: jax.Array, num_visits: int = None) -> dict:
@@ -49,19 +67,91 @@ def init_loop(cfg: LoCEDConfig, key: jax.Array, num_visits: int = None) -> dict:
     """
     NV = cfg.num_visits if num_visits is None else num_visits
     d = cfg.hidden_size
-    return dict(
+    out = dict(
         cond=jax.random.normal(key, (NV, d), jnp.float32) * cfg.init_std,
         gate_a=jnp.zeros((NV,), jnp.float32),               # sigmoid gates
         gate_m=jnp.zeros((NV,), jnp.float32),
         norm_f=jnp.ones((d,), jnp.float32),
     )
+    if cfg.mhc_streams > 1:
+        n = cfg.mhc_streams
+        # Diagonal-biased logits -> Sinkhorn starts close to identity.
+        res = jnp.zeros((NV, n, n), jnp.float32)
+        res = res.at[:, jnp.arange(n), jnp.arange(n)].set(1.0)
+        # Break stream symmetry: if H_pre/H_post were uniform and all streams
+        # started identical, H_res (row-stochastic) would keep them identical
+        # forever and mHC would collapse to a single stream.
+        bias = jnp.linspace(-0.2, 0.2, n, dtype=jnp.float32)
+        out.update(
+            mhc_res=res,
+            mhc_pre=jnp.broadcast_to(bias, (NV, n)),
+            mhc_post=jnp.broadcast_to(-bias, (NV, n)),
+        )
+        if cfg.mhc_dynamic:
+            out["mhc_dyn"] = jnp.zeros((NV, d, n * n + 2 * n), jnp.float32)
+    return out
+
+
+def _mhc_coefficients(cfg: LoCEDConfig, params: dict, x, visit: int):
+    """Return (H_res, H_pre, H_post) for one visit.
+
+    Static mode uses learnable per-visit logits. Dynamic mode predicts
+    per-token offsets from the mean stream (the single-pass coefficient idea),
+    then applies the same Birkhoff/softmax constraints.
+    """
+    n = cfg.mhc_streams
+    res = params["mhc_res"][visit]
+    pre = params["mhc_pre"][visit]
+    post = params["mhc_post"][visit]
+    if cfg.mhc_dynamic:
+        c = x.mean(axis=2) @ params["mhc_dyn"][visit]
+        res = res[None, None] + c[..., : n * n].reshape(c.shape[0], c.shape[1], n, n)
+        pre = pre[None, None] + c[..., n * n: n * n + n]
+        post = post[None, None] + c[..., n * n + n:]
+    return (sinkhorn(res, cfg.mhc_sinkhorn_iters),
+            jax.nn.softmax(pre, axis=-1),
+            jax.nn.softmax(post, axis=-1))
+
+
+def cell_visit_mhc(cfg: LoCEDConfig, params: dict, pc: dict, x, visit: int,
+                   attn_fn):
+    """One visit with mHC multi-stream residuals.
+
+    x: [B,T,n,d]. The sublayer still sees one [B,T,d] feature stream
+    (H_pre @ x); the update is injected back into all n streams through
+    H_post, and H_res mixes the streams. H_res is projected onto the
+    Birkhoff polytope by Sinkhorn, so residual propagation is non-expansive.
+    """
+    H_res, H_pre, H_post = _mhc_coefficients(cfg, params, x, visit)
+    if H_pre.ndim == 1:
+        pre = jnp.einsum("i,btid->btd", H_pre, x)
+    else:
+        pre = jnp.einsum("bti,btid->btd", H_pre, x)
+
+    h = rmsnorm(cfg, pre + params["cond"][visit].astype(pre.dtype), pc["norm_a"])
+    attn_out, ainfo = attn_fn(h)
+    update = jax.nn.sigmoid(params["gate_a"][visit]) * (attn_out @ pc["W_o"])
+    hm = rmsnorm(cfg, pre + update, pc["norm_m"])
+    mout, maux = moe_forward(cfg, pc, hm)
+    update = update + jax.nn.sigmoid(params["gate_m"][visit]) * mout
+
+    if H_res.ndim == 2:
+        mixed = jnp.einsum("ij,btjd->btid", H_res, x)
+        x_new = mixed + H_post[None, None, :, None] * update[:, :, None, :]
+    else:
+        mixed = jnp.einsum("btij,btjd->btid", H_res, x)
+        x_new = mixed + H_post[..., None] * update[:, :, None, :]
+    if cfg.magic_norm:
+        x_new = rmsnorm(cfg, x_new, pc["norm_magic"])
+    return x_new, h, ainfo, maux
 
 
 def cell_visit(cfg: LoCEDConfig, params: dict, pc: dict, x, visit: int,
                attn_fn):
     """One visit of ONE physical layer (`pc` = its own parameter dict).
 
-    x:        [B,T,d] residual state entering the visit.
+    x:        [B,T,d] residual state entering the visit, or [B,T,n,d] when
+              mHC is enabled.
     visit:    static visit index (loop-conditioning + gate selection; the
               layer index is implied by `pc`).
     attn_fn:  callable (h_normed [B,T,d]) -> (attention output [B,T,H*qd],
@@ -71,10 +161,15 @@ def cell_visit(cfg: LoCEDConfig, params: dict, pc: dict, x, visit: int,
     normed hidden (the caller reuses it as this visit's indexer query), and
     moe_aux is the router load-balance scalar of this visit.
     """
+    if cfg.mhc_streams > 1:
+        return cell_visit_mhc(cfg, params, pc, x, visit, attn_fn)
+
     h = rmsnorm(cfg, x + params["cond"][visit].astype(x.dtype), pc["norm_a"])
     attn_out, ainfo = attn_fn(h)
     x = x + jax.nn.sigmoid(params["gate_a"][visit]) * (attn_out @ pc["W_o"])
     hm = rmsnorm(cfg, x, pc["norm_m"])
     mout, maux = moe_forward(cfg, pc, hm)
     x = x + jax.nn.sigmoid(params["gate_m"][visit]) * mout
+    if cfg.magic_norm:
+        x = rmsnorm(cfg, x, pc["norm_magic"])
     return x, h, ainfo, maux
diff --git a/config.py b/config.py
index 0c03c37..2af46fb 100644
--- a/config.py
+++ b/config.py
@@ -29,6 +29,13 @@ class LoCEDConfig:
     max_seq_len: int = 512            # prefill length budget (prompt)
     max_decode_len: int = 128         # generated-token budget per session
 
+    # --- residual-stream architecture ---
+    # Both are opt-in so the frozen 100M/T=128 checkpoint path is unchanged.
+    magic_norm: bool = False          # module-level norm at each visit exit
+    mhc_streams: int = 1              # >1 enables Manifold-Constrained Hyper-Connections
+    mhc_sinkhorn_iters: int = 10      # Birkhoff projection iterations for H_res
+    mhc_dynamic: bool = False         # predict H_pre/H_post/H_res from the current state
+
     # --- attention ---
     num_attention_heads: int = 8
     head_dim: int = 32               # per-Q-head dim (NoPE throughout)
@@ -76,6 +83,12 @@ class LoCEDConfig:
     rms_norm_eps: float = 1e-6
     init_std: float = 0.02
 
+    def __post_init__(self):
+        if self.mhc_streams < 1:
+            raise ValueError("mhc_streams must be >= 1")
+        if self.mhc_sinkhorn_iters < 1:
+            raise ValueError("mhc_sinkhorn_iters must be >= 1")
+
     @property
     def num_layers(self) -> int:
         """Physical layers in the stack (4). Encode depth = pass-A depth."""
diff --git a/diagnostics.py b/diagnostics.py
index 63b38a1..fdc03f2 100644
--- a/diagnostics.py
+++ b/diagnostics.py
@@ -329,6 +329,9 @@ def run_baseline(steps: int = 150, batch: int = 4, seed: int = 0,
             "top_k": cfg.top_k,
             "pool_norm_preserve": cfg.pool_norm_preserve,
             "lazy_view": cfg.lazy_view,
+            "magic_norm": cfg.magic_norm,
+            "mhc_streams": cfg.mhc_streams,
+            "mhc_dynamic": cfg.mhc_dynamic,
             "train_delta": cfg.train_delta,
             "reuse_onpolicy": cfg.reuse_onpolicy,
         },
diff --git a/maxtext/config.py b/maxtext/config.py
index 573df51..e3a27c3 100644
--- a/maxtext/config.py
+++ b/maxtext/config.py
@@ -45,6 +45,10 @@ _LOCED_DEFAULTS: dict[str, Any] = {
     "loced_pool_m": 2,
     "loced_pool_norm_preserve": True,
     "loced_lazy_view": True,
+    "loced_magic_norm": False,
+    "loced_mhc_streams": 1,
+    "loced_mhc_sinkhorn_iters": 10,
+    "loced_mhc_dynamic": False,
     "loced_train_delta": True,
     "loced_train_prefill_frac": 0.5,
     "loced_reuse_onpolicy": True,
@@ -101,6 +105,18 @@ if MaxTextConfig is not None and Field is not None:
         loced_lazy_view: bool = Field(
             True, description="Score/gather candidate segments lazily (no [B,T,C,w] view)."
         )
+        loced_magic_norm: bool = Field(
+            False, description="Apply a module-level RMSNorm at every visit exit."
+        )
+        loced_mhc_streams: int = Field(
+            1, ge=1, description="mHC residual streams (1 disables mHC)."
+        )
+        loced_mhc_sinkhorn_iters: int = Field(
+            10, ge=1, description="Sinkhorn iterations for the mHC Birkhoff projection."
+        )
+        loced_mhc_dynamic: bool = Field(
+            False, description="Predict mHC mixing coefficients from the current state."
+        )
         loced_train_delta: bool = Field(
             True, description="Simulate the prompt->generated delta store during training (D3)."
         )
@@ -159,6 +175,10 @@ else:  # pragma: no cover - CPU-only fallback
         loced_pool_m: int = 2
         loced_pool_norm_preserve: bool = True
         loced_lazy_view: bool = True
+        loced_magic_norm: bool = False
+        loced_mhc_streams: int = 1
+        loced_mhc_sinkhorn_iters: int = 10
+        loced_mhc_dynamic: bool = False
         loced_train_delta: bool = True
         loced_train_prefill_frac: float = 0.5
         loced_reuse_onpolicy: bool = True
@@ -262,6 +282,10 @@ def loced_config_from_maxtext(config: Any):
         pool_m=int(_loced_get(config, "loced_pool_m")),
         pool_norm_preserve=bool(_loced_get(config, "loced_pool_norm_preserve")),
         lazy_view=bool(_loced_get(config, "loced_lazy_view")),
+        magic_norm=bool(_loced_get(config, "loced_magic_norm")),
+        mhc_streams=int(_loced_get(config, "loced_mhc_streams")),
+        mhc_sinkhorn_iters=int(_loced_get(config, "loced_mhc_sinkhorn_iters")),
+        mhc_dynamic=bool(_loced_get(config, "loced_mhc_dynamic")),
         train_delta=bool(_loced_get(config, "loced_train_delta")),
         train_prefill_frac=float(_loced_get(config, "loced_train_prefill_frac")),
         reuse_onpolicy=bool(_loced_get(config, "loced_reuse_onpolicy")),
@@ -358,6 +382,10 @@ def loced_config_from_mapping(mapping: Mapping[str, Any]):
         pool_m=int(get("loced_pool_m", base.pool_m)),
         pool_norm_preserve=bool(get("loced_pool_norm_preserve", base.pool_norm_preserve)),
         lazy_view=bool(get("loced_lazy_view", base.lazy_view)),
+        magic_norm=bool(get("loced_magic_norm", base.magic_norm)),
+        mhc_streams=int(get("loced_mhc_streams", base.mhc_streams)),
+        mhc_sinkhorn_iters=int(get("loced_mhc_sinkhorn_iters", base.mhc_sinkhorn_iters)),
+        mhc_dynamic=bool(get("loced_mhc_dynamic", base.mhc_dynamic)),
         train_delta=bool(get("loced_train_delta", base.train_delta)),
         train_prefill_frac=float(get("loced_train_prefill_frac", base.train_prefill_frac)),
         reuse_onpolicy=bool(get("loced_reuse_onpolicy", base.reuse_onpolicy)),
diff --git a/maxtext/modeling.py b/maxtext/modeling.py
index 8f578e4..884e25d 100644
--- a/maxtext/modeling.py
+++ b/maxtext/modeling.py
@@ -160,11 +160,13 @@ class LoCEDLayer(nnx.Module):
         d = cfg.hidden_size
         self.norm_a = _p(jnp.ones((d,), jnp.float32), ("norm",))
         self.norm_m = _p(jnp.ones((d,), jnp.float32), ("norm",))
+        if cfg.magic_norm:
+            self.norm_magic = _p(jnp.ones((d,), jnp.float32), ("norm",))
         self.attn = LoCEDAttention(cfg, rngs)
         self.moe = LoCEDMoE(cfg, rngs)
 
     def reference_dict(self) -> dict:
-        return {
+        out = {
             "norm_a": self.norm_a.get_value(),
             "norm_m": self.norm_m.get_value(),
             "W_q": self.attn.W_q.get_value(),
@@ -173,6 +175,9 @@ class LoCEDLayer(nnx.Module):
             "W_o": self.attn.W_o.get_value(),
             **self.moe.reference_dict(),
         }
+        if hasattr(self, "norm_magic"):
+            out["norm_magic"] = self.norm_magic.get_value()
+        return out
 
 
 class LoCEDModel(nnx.Module):
@@ -217,6 +222,22 @@ class LoCEDModel(nnx.Module):
         self.gate_m = _p(jnp.zeros((nv,), jnp.float32), (None,))
         self.norm_f = _p(jnp.ones((d,), jnp.float32), ("norm",))
 
+        if cfg.mhc_streams > 1:
+            n = cfg.mhc_streams
+            res = jnp.zeros((nv, n, n), jnp.float32)
+            res = res.at[:, jnp.arange(n), jnp.arange(n)].set(1.0)
+            bias = jnp.linspace(-0.2, 0.2, n, dtype=jnp.float32)
+            self.mhc_res = _p(res, (None, None, None))
+            self.mhc_pre = _p(
+                jnp.broadcast_to(bias, (nv, n)), (None, None))
+            self.mhc_post = _p(
+                jnp.broadcast_to(-bias, (nv, n)), (None, None))
+            if cfg.mhc_dynamic:
+                self.mhc_dyn = _p(
+                    jnp.zeros((nv, d, n * n + 2 * n), jnp.float32),
+                    (None, None, None),
+                )
+
         # Use an NNX dict with string keys rather than nnx.List: MaxText's
         # Muon path classifier (muon_utils.transform_logic) expects every
         # path segment to be a string.  nnx.List lowers to integer dict keys
@@ -230,7 +251,7 @@ class LoCEDModel(nnx.Module):
     # ------------------------------------------------------------------
     def to_reference_params(self) -> dict:
         """Assemble the exact pytree consumed by ``loced.model``."""
-        return {
+        out = {
             "embed": self.token_embedding.get_value(),
             "head": self.logits_dense.get_value(),
             "W_bk": self.W_bk.get_value(),
@@ -243,6 +264,15 @@ class LoCEDModel(nnx.Module):
             "norm_f": self.norm_f.get_value(),
             "cells": tuple(cell.reference_dict() for cell in self.cells.values()),
         }
+        if getattr(self, "mhc_res", None) is not None:
+            out.update(
+                mhc_res=self.mhc_res.get_value(),
+                mhc_pre=self.mhc_pre.get_value(),
+                mhc_post=self.mhc_post.get_value(),
+            )
+            if getattr(self, "mhc_dyn", None) is not None:
+                out["mhc_dyn"] = self.mhc_dyn.get_value()
+        return out
 
     def load_reference_params(self, params: dict) -> "LoCEDModel":
         """Copy a reference pytree into the NNX parameter graph."""
@@ -256,9 +286,17 @@ class LoCEDModel(nnx.Module):
         self.gate_a.set_value(params["gate_a"])
         self.gate_m.set_value(params["gate_m"])
         self.norm_f.set_value(params["norm_f"])
+        if getattr(self, "mhc_res", None) is not None:
+            self.mhc_res.set_value(params["mhc_res"])
+            self.mhc_pre.set_value(params["mhc_pre"])
+            self.mhc_post.set_value(params["mhc_post"])
+            if getattr(self, "mhc_dyn", None) is not None:
+                self.mhc_dyn.set_value(params["mhc_dyn"])
         for cell, src in zip(self.cells.values(), params["cells"]):
             cell.norm_a.set_value(src["norm_a"])
             cell.norm_m.set_value(src["norm_m"])
+            if hasattr(cell, "norm_magic"):
+                cell.norm_magic.set_value(src["norm_magic"])
             cell.attn.W_q.set_value(src["W_q"])
             cell.attn.W_fx.set_value(src["W_fx"])
             cell.attn.W_fv.set_value(src["W_fv"])
@@ -317,10 +355,11 @@ class LoCEDModel(nnx.Module):
         supervision term; ``indexer_loss`` and ``moe_lb_loss`` use MaxText's
         standard suffix names so MaxText's own ``loss_fn`` can pick them up.
         """
+        reuse_p = kwargs.pop("reuse_p", 0.0)
         del decoder_segment_ids, enable_dropout, kwargs
         tokens = decoder_input_tokens
         if model_mode == MODEL_MODE_TRAIN:
-            out = self.train_output(tokens)
+            out = self.train_output(tokens, reuse_p=reuse_p)
             self.sow(nnx.Intermediate, "loced_loss", out["loss"])
             self.sow(nnx.Intermediate, "loced_ce", out["ce"])
             self.sow(nnx.Intermediate, "loced_ce_aux", out["ce_aux"])
diff --git a/model.py b/model.py
index 341a29f..cd4e372 100644
--- a/model.py
+++ b/model.py
@@ -87,13 +87,27 @@ def count_params(params) -> int:
 # shared visit primitives
 # ---------------------------------------------------------------------------
 
+def _streams(cfg, x):
+    """Lift [B,T,d] to [B,T,n,d] when mHC is enabled."""
+    if cfg.mhc_streams == 1 or x.ndim == 4:
+        return x
+    return jnp.broadcast_to(
+        x[..., None, :], x.shape[:2] + (cfg.mhc_streams, x.shape[-1]))
+
+
+def _merge(cfg, x):
+    """Collapse mHC streams to the single feature stream used by the head,
+    bridge, indexer, and KV store."""
+    return x if x.ndim == 3 else jnp.mean(x, axis=2)
+
+
 def _q(cfg, pc, h):
     B, T, _ = h.shape
     return (h @ pc["W_q"]).reshape(B, T, cfg.num_attention_heads, cfg.head_dim)
 
 
 def _head(cfg, p, x):
-    return rmsnorm(cfg, x, p["norm_f"]) @ p["head"]
+    return rmsnorm(cfg, _merge(cfg, x), p["norm_f"]) @ p["head"]
 
 
 def _ce(logits, targets):
@@ -115,6 +129,7 @@ def _gather_valid(valid, idx):
 
 def _bridge(cfg, params, H):
     """KV Bridging: hidden states -> shared store entries (k, v, indexer-K)."""
+    H = _merge(cfg, H)
     k = H @ params["W_bk"]
     v = H @ params["W_bv"]
     ik = jax.lax.stop_gradient(k) @ params["W_ik"]
@@ -179,6 +194,7 @@ def _encode_pass(cfg, params, x, positions):
     """
     info = dict()
     maux = jnp.asarray(0.0, jnp.float32)
+    x = _streams(cfg, x)
     for k, mode in enumerate(cfg.encode_layers):
         if mode != "full":
             raise ValueError(f"encode layer {k} is '{mode}', expected 'full'")
@@ -200,6 +216,7 @@ def _decode_pass(cfg, params, x, Kg, Vg, vg, view, oracle=None,
     maux = jnp.asarray(0.0, jnp.float32)
     kls = []
     infos = []
+    x = _streams(cfg, x)
     for k, mode in enumerate(cfg.sparse_layers):
         v = V + k
         pc = params["cells"][k]
@@ -226,7 +243,7 @@ def _decode_pass(cfg, params, x, Kg, Vg, vg, view, oracle=None,
 def _anchor_select(cfg, params, x, view):
     """Inference anchor: indexer scores at the exit hidden pick the rows the
     REUSE layers inherit (top-k + forced window)."""
-    lg = indexer_scores_view(cfg, params, x, view)
+    lg = indexer_scores_view(cfg, params, _merge(cfg, x), view)
     idx = select(lg, view["valid"], view["boost"], _L(cfg, view))
     Kg, Vg = _gather(view, idx)
     vg = _gather_valid(view["valid"], idx)
@@ -301,7 +318,8 @@ def _forward_train_impl(cfg: LoCEDConfig, params: dict, tokens,
     # REUSE mixes in the indexer-selected rows (1.3) so training matches the
     # inference selector. `reuse_p=0` keeps the original oracle path.
     idx_a = select(oracle, view["valid"], view["boost"], _L(cfg, view))
-    lg_a = indexer_scores_view(cfg, params, jax.lax.stop_gradient(H_out),
+    lg_a = indexer_scores_view(cfg, params,
+                               jax.lax.stop_gradient(_merge(cfg, H_out)),
                                view)
     kl_a = kl_distill(oracle, lg_a, view["valid"])
     idx_i = (select(lg_a, view["valid"], view["boost"], _L(cfg, view))
@@ -362,7 +380,8 @@ def _forward_train_impl(cfg: LoCEDConfig, params: dict, tokens,
                logits=logits)
     if diagnostics:
         out.update(
-            H_out=H_out, probs_mean=probs_mean, view=view, oracle=oracle,
+            H_out=_merge(cfg, H_out), probs_mean=probs_mean, view=view,
+            oracle=oracle,
             idx_oracle=idx_a, idx_indexer=idx_i, anchor_logits=lg_a,
             pool_k=pool_k, per_token_k=pk, pool_last=pool_last,
             visit_infos=infos)
@@ -406,7 +425,7 @@ def prefill(cfg: LoCEDConfig, params: dict, tokens):
     store = new_store(cfg, B, params["embed"].dtype)
 
     H_out, _, _ = _encode_pass(cfg, params, x, positions)
-    store = build_pool(cfg, store, H_out, positions,
+    store = build_pool(cfg, store, _merge(cfg, H_out), positions,
                        params["W_bk"], params["W_bv"], params["W_ik"])
     pk, pv, pik = _bridge(cfg, params, H_out)
     W = cfg.window
@@ -433,7 +452,7 @@ def prefill_extend(cfg: LoCEDConfig, params: dict, store: dict, tokens,
     x = params["embed"][tokens]
 
     H_out, _, _ = _encode_pass(cfg, params, x, positions)
-    store = build_pool(cfg, store, H_out, positions,
+    store = build_pool(cfg, store, _merge(cfg, H_out), positions,
                        params["W_bk"], params["W_bv"], params["W_ik"])
     pk, pv, pik = _bridge(cfg, params, H_out)
     W, n = cfg.window, min(T, cfg.window)
diff --git a/smoke_test.py b/smoke_test.py
index 1e38acc..bcab444 100644
--- a/smoke_test.py
+++ b/smoke_test.py
@@ -13,6 +13,7 @@ import jax.numpy as jnp
 import numpy as np
 import pytest
 
+from .block import sinkhorn
 from .cache import new_store
 from .config import LoCEDConfig
 from .diagnostics import (build_report, pool_norm_ratio, segment_layout,
@@ -464,6 +465,52 @@ def test_prefill_extend_lazy_matches_dense(params):
                                    rtol=1e-5, atol=1e-5)
 
 
+def test_magic_norm(params):
+    """MagicNorm caps every visit with a module-level RMSNorm."""
+    cfg = dataclasses.replace(CFG, magic_norm=True)
+    p = init_model(cfg, KEY)
+    assert all("norm_magic" in cell for cell in p["cells"])
+    assert count_params(p) - count_params(params) == CFG.num_layers * CFG.hidden_size
+    tokens = _diag_batch(2, 32)
+    out = forward_train(cfg, p, tokens)
+    assert np.isfinite(float(out["loss"]))
+    assert out["logits"].shape == (2, 32, CFG.vocab_size)
+
+
+def test_mhc_streams_and_sinkhorn(params):
+    """mHC residual streams stay finite and H_res is doubly stochastic."""
+    cfg = dataclasses.replace(CFG, mhc_streams=4, magic_norm=True)
+    p = init_model(cfg, KEY)
+    assert p["mhc_res"].shape == (CFG.num_visits, 4, 4)
+    h = sinkhorn(p["mhc_res"], cfg.mhc_sinkhorn_iters)
+    np.testing.assert_allclose(np.asarray(h.sum(axis=-1)), 1.0, atol=1e-5)
+    np.testing.assert_allclose(np.asarray(h.sum(axis=-2)), 1.0, atol=1e-5)
+
+    tokens = _diag_batch(2, 32)
+    out = forward_train(cfg, p, tokens)
+    assert np.isfinite(float(out["loss"]))
+    # Streams must actually diverge; a uniform H_pre/H_post would collapse mHC
+    # back to a single residual stream.
+    from .model import _encode_pass
+    positions = jnp.broadcast_to(jnp.arange(32)[None, :], tokens.shape)
+    streams, _, _ = _encode_pass(cfg, p, p["embed"][tokens], positions)
+    assert float(np.ptp(np.asarray(streams), axis=2).mean()) > 1e-4
+    store, logits = prefill(cfg, p, tokens[:, :16])
+    assert logits.shape == (2, 1, CFG.vocab_size)
+    store, logits = decode_step(cfg, p, store, tokens[:, 16], 16)
+    assert logits.shape == (2, 1, CFG.vocab_size)
+
+
+def test_mhc_dynamic(params):
+    """The single-pass dynamic-coefficient path is differentiable and finite."""
+    cfg = dataclasses.replace(CFG, mhc_streams=4, mhc_dynamic=True)
+    p = init_model(cfg, KEY)
+    assert p["mhc_dyn"].shape == (
+        CFG.num_visits, CFG.hidden_size, 4 * 4 + 2 * 4)
+    out = forward_train(cfg, p, _diag_batch(2, 32))
+    assert np.isfinite(float(out["loss"]))
+
+
 def test_lazy_view_matches_materialized(params):
     """M3.1: segment scoring/gather matches the materialized candidate view."""
     tokens = _diag_batch(2, 32)
'''

# ---------------------------------------------------------------------------
# User knobs
# ---------------------------------------------------------------------------
ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "tpu")  # "tpu" or "gpu"
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
MAXTEXT_EXTRAS = "cuda12" if ACCELERATOR == "gpu" else "tpu"

LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git"
)
# This branch contains commits 2191acf and e0b29f9.
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "architecture-mhc-magicnorm")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
RUNS = WORK / "loced_runs"
CACHE = WORK / "jax_cache"
for _p in (SCRIPTS, RUNS, CACHE):
    _p.mkdir(parents=True, exist_ok=True)


def _loced_parent(root: Path):
    # Return the directory to put on PYTHONPATH for `import loced`.
    if (root / "loced" / "maxtext" / "train.py").is_file():
        return root
    # offx-zinth/loced is the package contents at the repo root.
    if root.name == "loced" and (root / "maxtext" / "train.py").is_file():
        return root.parent
    return None


def _find_loced_root():
    candidates = []
    if os.environ.get("LOCED_ROOT"):
        candidates.append(Path(os.environ["LOCED_ROOT"]))
    candidates += [
        WORK,
        WORK / "loced",
        Path.cwd(),
        Path.cwd() / "loced",
        Path("/kaggle/input"),
    ]
    for root in candidates:
        found = _loced_parent(root)
        if found is not None:
            return found
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.glob("**/maxtext/train.py"):
            repo = p.parents[1]
            if repo.name == "loced" or (
                (repo / "attention.py").is_file()
                and (repo / "config.py").is_file()
            ):
                return repo.parent
        for p in kaggle_input.glob("**/loced/maxtext/train.py"):
            return p.parents[2]
    return None


# Keep a reused Kaggle working directory in sync with the branch under test.
# If the branch is not on GitHub yet, fall back to master + the embedded patch.
_target = WORK / "loced"
APPLY_ARCH_PATCH = os.environ.get("LOCED_APPLY_ARCH_PATCH", "auto")
LOCED_REF_USED = LOCED_REPO_REF


def _git(env, *args, check=True):
    return subprocess.run(
        ["git", *args], check=check, env=env,
        text=True, capture_output=True,
    )


def _apply_arch_patch(target: Path):
    cfg_text = (target / "config.py").read_text()
    if "mhc_streams" in cfg_text and "magic_norm" in cfg_text:
        # master already contains the architecture commits; nothing to apply.
        print("MagicNorm/mHC already present; skipping embedded patch")
        return
    patch_path = WORK / "loced_arch.patch"
    patch_path.write_text(ARCH_PATCH)
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"applying embedded architecture patch -> {target}")
    _git(env, "-C", str(target), "apply", "--whitespace=nowarn", str(patch_path))
    cfg_text = (target / "config.py").read_text()
    if "mhc_streams" not in cfg_text or "magic_norm" not in cfg_text:
        raise RuntimeError("embedded patch did not add MagicNorm/mHC config")
    print("architecture patch applied")


def _fetch_ref(target: Path, ref: str):
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    _git(env, "-C", str(target), "fetch", "--depth", "1", "origin", ref)
    _git(env, "-C", str(target), "reset", "--hard", "FETCH_HEAD")


def _clone_ref(target: Path, ref: str):
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"cloning {LOCED_REPO_URL} ({ref}) -> {target}")
    _git(env, "clone", "--depth", "1", "--branch", ref,
         LOCED_REPO_URL, str(target))


if LOCED_REPO_URL and (_target / ".git").is_dir():
    print(f"updating existing checkout {_target} ({LOCED_REPO_REF})")
    try:
        _fetch_ref(_target, LOCED_REPO_REF)
    except subprocess.CalledProcessError:
        if APPLY_ARCH_PATCH not in ("auto", "1"):
            raise
        print(f"{LOCED_REPO_REF} is not on origin; using master fallback")
        _fetch_ref(_target, "master")
        _apply_arch_patch(_target)
        LOCED_REF_USED = "master+architecture-patch"

LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None and LOCED_REPO_URL:
    target = WORK / "loced"
    if not (target / ".git").is_dir():
        if target.is_dir() and any(target.iterdir()):
            print(f"{target} exists without .git; treating it as the source")
        else:
            try:
                _clone_ref(target, LOCED_REPO_REF)
                LOCED_REF_USED = LOCED_REPO_REF
            except subprocess.CalledProcessError:
                if APPLY_ARCH_PATCH not in ("auto", "1"):
                    raise
                print(f"{LOCED_REPO_REF} is not on origin; using master fallback")
                shutil.rmtree(target, ignore_errors=True)
                _clone_ref(target, "master")
                _apply_arch_patch(target)
                LOCED_REF_USED = "master+architecture-patch"
    LOCED_ROOT = _find_loced_root()

print(f"ACCELERATOR : {ACCELERATOR}")
print(f"WORK        : {WORK}")
print(f"LOCED_ROOT  : {LOCED_ROOT}")
print(f"LOCED_REF   : {LOCED_REF_USED}")
print(f"MAXTEXT_DIR : {MAXTEXT_DIR}")
print(f"VENV        : {VENV_DIR}")
if LOCED_ROOT is None:
    raise RuntimeError(textwrap.dedent(f'''
        LoCED source not found.

        Push the branch to GitHub and clone it, or upload the repository as a
        Kaggle Dataset:

          LOCED_REPO_URL = {LOCED_REPO_URL!r}
          LOCED_REPO_REF = {LOCED_REPO_REF!r}

        The checkout must contain `loced/maxtext/train.py`.
    '''))
print("nvidia-smi:")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)
else:
    print("nvidia-smi not found (TPU or CPU kernel)")

## 1. Python 3.12 + MaxText v0.2.5

MaxText v0.2.5 needs Python >= 3.12. The notebook creates an isolated `uv`
venv and installs the MaxText TPU/GPU extras plus pytest.

Set `LOCED_SKIP_INSTALL=1` only when re-running against an already prepared
venv and MaxText checkout.

In [ ]:
SKIP_INSTALL = os.environ.get("LOCED_SKIP_INSTALL", "0") == "1"

if SKIP_INSTALL:
    print("LOCED_SKIP_INSTALL=1: using existing venv/MaxText checkout")
else:
    if shutil.which("uv") is None:
        print("installing uv ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv is not on PATH after pip install")

    if not PY.exists():
        print("creating Python 3.12 venv ...")
        subprocess.run(
            [UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)],
            check=True,
        )
    print(subprocess.check_output([str(PY), "--version"], text=True).strip())

    if not (MAXTEXT_DIR / "pyproject.toml").is_file():
        print(f"cloning MaxText {MAXTEXT_REF} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
             MAXTEXT_REPO, str(MAXTEXT_DIR)],
            check=True,
        )
    else:
        print("MaxText checkout already present")

    print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
        cwd=str(MAXTEXT_DIR), check=True,
    )
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "pytest", "pytest-timeout"],
        check=True,
    )
    print("dependency check:")
    subprocess.run([UV, "pip", "check", "--python", str(PY)], check=False)

print(subprocess.check_output(
    [str(PY), "-c",
     "import sys, maxtext; print(sys.version); print(maxtext.__file__)"],
    text=True,
))

## 2. TPU/GPU + real-MaxText sanity check

This is deliberately strict:

- JAX must see the requested accelerator;
- MaxText's real utilities must import (no silent CPU stubs);
- the checked-out branch must actually contain the MagicNorm/mHC code.

The last check prevents a green run against an old `master` checkout.

In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_ROOT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.90",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "true",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})

# jax-cuda12 loads pip-installed NVIDIA libraries with ctypes. In a venv they
# are under site-packages/nvidia/*/lib, which is not on the default loader path.
_nvidia_code = (
    "import site; from pathlib import Path; "
    "print(':'.join(str(p) for r in site.getsitepackages() "
    "for p in sorted((Path(r) / 'nvidia').glob('*/lib')) if p.is_dir()))"
)
_nvidia_libs = subprocess.check_output([str(PY), "-c", _nvidia_code], text=True).strip()
if _nvidia_libs:
    BASE_ENV["LD_LIBRARY_PATH"] = ":".join(
        x for x in (_nvidia_libs, os.environ.get("LD_LIBRARY_PATH", "")) if x
    )
print("NVIDIA lib dirs:", len(_nvidia_libs.split(":")) if _nvidia_libs else 0)


def run_script(name, source, timeout=3600, extra_env=None, check=True, cwd=None):
    path = SCRIPTS / name
    path.write_text(textwrap.dedent(source))
    env = BASE_ENV.copy()
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    t0 = time.time()
    proc = subprocess.run(
        [str(PY), str(path)],
        cwd=str(cwd or LOCED_ROOT),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - t0
    print(proc.stdout)
    if proc.stderr.strip():
        print("[stderr]\n" + proc.stderr[-12000:])
    print(f"[{name}] exit={proc.returncode} elapsed={elapsed:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"{name} failed (exit {proc.returncode})")
    return proc


sanity = r'''
import json
import os
import subprocess
import sys

import jax
import maxtext

print("python  :", sys.version.split()[0])
print("jax     :", jax.__version__)
print("maxtext :", maxtext.__file__)
print("backend :", jax.default_backend())
print("devices :", jax.devices())

expected = os.environ["LOCED_EXPECTED_BACKEND"]
assert jax.default_backend() == expected, (
    f"Expected {expected}, got {jax.default_backend()}. "
    "Set the Kaggle accelerator and restart the session."
)
assert len(jax.devices()) >= 1
for d in jax.devices():
    stats = d.memory_stats() or {}
    print(f"  {d}: {d.device_kind} bytes_limit={stats.get('bytes_limit')}")

# Fail if MaxText silently fell back to the CPU-stub route used by local tests.
import maxtext.utils.maxtext_utils  # noqa: F401
import maxtext.utils.model_creation_utils  # noqa: F401
print("real MaxText utilities: OK")

from loced.block import sinkhorn
from loced.config import LoCEDConfig

cfg = LoCEDConfig()
assert cfg.magic_norm is False
assert cfg.mhc_streams == 1
assert hasattr(sinkhorn, "__call__")
print("MagicNorm/mHC symbols: OK")

repo = os.environ["LOCED_ROOT"]
for cmd in (
    ["git", "-C", repo, "rev-parse", "--short", "HEAD"],
    ["git", "-C", repo, "log", "-3", "--oneline"],
    ["git", "-C", repo, "status", "--short"],
):
    proc = subprocess.run(cmd, text=True, capture_output=True)
    print(proc.stdout.strip())
'''
run_script(
    "sanity.py", sanity, timeout=1200,
    extra_env={
        "LOCED_EXPECTED_BACKEND": ACCELERATOR,
        "LOCED_ROOT": str(LOCED_ROOT),
    },
)

## 3. Reference smoke suite

Runs `loced/smoke_test.py`, including the new MagicNorm, static mHC, dynamic
mHC, Sinkhorn, and stream-divergence tests.

In [ ]:
REF_LOG = WORK / "reference_pytest.log"


def run_pytest(name, target, extra_env=None, timeout=3600):
    env = BASE_ENV.copy()
    env.update({
        "LOCED_ACCELERATOR": ACCELERATOR,
        "LOCED_ROOT": str(LOCED_ROOT),
    })
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    cmd = [
        str(PY), "-m", "pytest", "-q", target,
        "-x", "--durations=10", "--timeout=1800", "-p", "no:cacheprovider",
    ]
    t0 = time.time()
    proc = subprocess.run(
        cmd, cwd=str(LOCED_ROOT), env=env,
        text=True, capture_output=True, timeout=timeout,
    )
    print(proc.stdout)
    if proc.stderr.strip():
        print("[stderr]\n" + proc.stderr[-12000:])
    print(f"{name} exit={proc.returncode} elapsed={time.time() - t0:.1f}s")
    log = WORK / f"{name}.log"
    log.write_text(proc.stdout + "\n[stderr]\n" + proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"{name} failed (exit {proc.returncode}); see {log}")
    return proc, log


ref_proc, ref_log = run_pytest(
    "reference_pytest", "loced/smoke_test.py", timeout=5400,
)
ref_match = re.search(r"(\d+) passed", ref_proc.stdout)
assert ref_match, "could not parse reference pytest result"
ref_report = {
    "passed": int(ref_match.group(1)),
    "exit_code": ref_proc.returncode,
    "log": str(ref_log),
}
(WORK / "reference_test_report.json").write_text(json.dumps(ref_report, indent=2))
print("reference tests:", ref_report)

## 4. MaxText integration suite

Runs `loced/maxtext/test_integration.py` with:

- `LOCED_REQUIRE_REAL_MAXTEXT_UTILS=1` — no CPU-stub fallback;
- `LOCED_PARITY_TOL=1e-5` — TPU/GPU fusion tolerance.

This covers the default (single-stream, no MagicNorm) MaxText path: parameter
inventory, loss/logits, gradients, prefill/decode, Muon, sharding, config, and
checkpoint/resume.

In [ ]:
maxtext_proc, maxtext_log = run_pytest(
    "maxtext_pytest",
    "loced/maxtext/test_integration.py",
    extra_env={
        "LOCED_PARITY_TOL": "1e-5",
        "LOCED_REQUIRE_REAL_MAXTEXT_UTILS": "1",
    },
    timeout=7200,
)
maxtext_match = re.search(r"(\d+) passed", maxtext_proc.stdout)
assert maxtext_match, "could not parse MaxText pytest result"
maxtext_report = {
    "passed": int(maxtext_match.group(1)),
    "exit_code": maxtext_proc.returncode,
    "parity_tolerance": 1e-5,
    "log": str(maxtext_log),
}
(WORK / "maxtext_test_report.json").write_text(json.dumps(maxtext_report, indent=2))
print("MaxText tests:", maxtext_report)

## 5. Reference ↔ NNX parity for MagicNorm / mHC

Four configurations are compared:

| name | MagicNorm | streams | dynamic coefficients |
|---|---:|---:|---:|
| `baseline` | off | 1 | off |
| `magic_norm` | on | 1 | off |
| `mhc_static_magic` | on | 4 | off |
| `mhc_dynamic_magic` | on | 4 | on |

For each one, the script checks parameter identity, `reuse_p=0.5` loss/logits,
gradients, prefill store/logits, and two decode steps. It also checks that the
`H_res` matrix is doubly stochastic and that the four streams actually diverge.

In [ ]:
parity = r'''
import dataclasses
import json
import os

import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx

from loced.block import sinkhorn
from loced.model import (
    _encode_pass, count_params, decode_step, forward_train, init_model, prefill,
)
from loced.train import synthetic_batch, tiny_config
from loced.maxtext.modeling import build_model

TOL = float(os.environ.get("LOCED_PARITY_TOL", "1e-5"))
REUSE_P = 0.5


def maxdiff(a, b):
    return float(jnp.max(jnp.abs(jnp.asarray(a) - jnp.asarray(b))))


def map_grad(pure, cfg):
    cells = []
    for i in range(cfg.num_layers):
        src = pure["cells"][str(i)]
        cell = {
            "norm_a": src["norm_a"],
            "norm_m": src["norm_m"],
            "W_q": src["attn"]["W_q"],
            "W_fx": src["attn"]["W_fx"],
            "W_fv": src["attn"]["W_fv"],
            "W_o": src["attn"]["W_o"],
            "W_gate": src["moe"]["routed_moe"]["gate"],
            "Ws1": src["moe"]["shared_experts"]["w1"],
            "Ws3": src["moe"]["shared_experts"]["w3"],
            "Ws2": src["moe"]["shared_experts"]["w2"],
            "We1": src["moe"]["routed_moe"]["wi_0"],
            "We3": src["moe"]["routed_moe"]["wi_1"],
            "We2": src["moe"]["routed_moe"]["wo"],
        }
        if cfg.magic_norm:
            cell["norm_magic"] = src["norm_magic"]
        cells.append(cell)
    out = {
        "embed": pure["token_embedding"],
        "head": pure["logits_dense"],
        "W_bk": pure["W_bk"],
        "W_bv": pure["W_bv"],
        "W_ik": pure["W_ik"],
        "W_iq": pure["W_iq"],
        "cond": pure["visit_embedding"],
        "gate_a": pure["gate_a"],
        "gate_m": pure["gate_m"],
        "norm_f": pure["norm_f"],
        "cells": tuple(cells),
    }
    if cfg.mhc_streams > 1:
        out.update(
            mhc_res=pure["mhc_res"],
            mhc_pre=pure["mhc_pre"],
            mhc_post=pure["mhc_post"],
        )
        if cfg.mhc_dynamic:
            out["mhc_dyn"] = pure["mhc_dyn"]
    return out


base = tiny_config()
configs = {
    "baseline": base,
    "magic_norm": dataclasses.replace(base, magic_norm=True),
    "mhc_static_magic": dataclasses.replace(
        base, magic_norm=True, mhc_streams=4),
    "mhc_dynamic_magic": dataclasses.replace(
        base, magic_norm=True, mhc_streams=4, mhc_dynamic=True),
}
tokens = synthetic_batch(
    np.random.default_rng(7), 2, base.max_seq_len, base.vocab_size)
report = {
    "tolerance": TOL,
    "reuse_p": REUSE_P,
    "backend": jax.default_backend(),
    "devices": len(jax.devices()),
    "configs": {},
}

for name, cfg in configs.items():
    print(f"\n=== {name} ===", flush=True)
    ref = init_model(cfg, jax.random.PRNGKey(0))
    model = build_model(cfg, seed=0, reference_params=ref)
    assert model.count_params() == count_params(ref), (
        "parameter count mismatch", model.count_params(), count_params(ref))

    pure = model.to_reference_params()
    assert jax.tree.structure(pure) == jax.tree.structure(ref), (
        "parameter tree mismatch")
    param_diff = max(
        maxdiff(a, b) for a, b in zip(
            jax.tree.leaves(pure), jax.tree.leaves(ref)))
    assert param_diff == 0.0, f"parameter values differ: {param_diff}"

    ref_out = forward_train(cfg, ref, tokens, reuse_p=REUSE_P)
    got_out = model.train_output(tokens, reuse_p=REUSE_P)
    loss_diff = maxdiff(ref_out["loss"], got_out["loss"])
    logits_diff = maxdiff(ref_out["logits"], got_out["logits"])
    assert loss_diff <= TOL, f"loss maxdiff {loss_diff}"
    assert logits_diff <= TOL, f"logits maxdiff {logits_diff}"

    ref_grad = jax.grad(
        lambda p: forward_train(cfg, p, tokens, reuse_p=REUSE_P)["loss"]
    )(ref)
    nnx_grad = nnx.grad(
        lambda m: m.train_loss(tokens, reuse_p=REUSE_P)[0]
    )(model)
    mapped_grad = map_grad(nnx.to_pure_dict(nnx_grad), cfg)
    assert jax.tree.structure(mapped_grad) == jax.tree.structure(ref_grad), (
        "gradient tree mismatch")
    grad_diff = max(
        maxdiff(a, b) for a, b in zip(
            jax.tree.leaves(mapped_grad), jax.tree.leaves(ref_grad)))
    assert grad_diff <= TOL, f"gradient maxdiff {grad_diff}"

    ref_store, ref_prefill_logits = prefill(cfg, ref, tokens)
    got_store, got_prefill_logits = model.prefill_tokens(tokens)
    prefill_logits_diff = maxdiff(ref_prefill_logits, got_prefill_logits)
    assert prefill_logits_diff <= TOL, (
        f"prefill logits maxdiff {prefill_logits_diff}")
    store_diff = max(
        maxdiff(a, b) for a, b in zip(
            jax.tree.leaves(ref_store), jax.tree.leaves(got_store)))
    assert store_diff <= TOL, f"prefill store maxdiff {store_diff}"

    decode_diffs = []
    for i in range(2):
        token = jnp.asarray([3, 7], jnp.int32)
        pos = cfg.max_seq_len + i
        ref_store, ref_step = decode_step(cfg, ref, ref_store, token, pos)
        got_store, got_step = model.decode_token(got_store, token, pos)
        d = maxdiff(ref_step, got_step)
        decode_diffs.append(d)
        assert d <= TOL, f"decode step {i} maxdiff {d}"

    sink_err = None
    stream_ptp = None
    if cfg.mhc_streams > 1:
        h = np.asarray(sinkhorn(ref["mhc_res"], cfg.mhc_sinkhorn_iters))
        row_err = float(np.max(np.abs(h.sum(axis=-1) - 1.0)))
        col_err = float(np.max(np.abs(h.sum(axis=-2) - 1.0)))
        sink_err = max(row_err, col_err)
        assert sink_err < 1e-4, f"Sinkhorn error {sink_err}"
        positions = jnp.broadcast_to(
            jnp.arange(tokens.shape[1])[None, :], tokens.shape)
        streams, _, _ = _encode_pass(
            cfg, ref, ref["embed"][tokens], positions)
        stream_ptp = float(np.ptp(np.asarray(streams), axis=2).mean())
        assert stream_ptp > 1e-4, (
            f"mHC streams collapsed to one stream: ptp {stream_ptp}")

    report["configs"][name] = {
        "params": int(count_params(ref)),
        "param_maxdiff": param_diff,
        "loss_maxdiff": loss_diff,
        "logits_maxdiff": logits_diff,
        "grad_maxdiff": grad_diff,
        "prefill_logits_maxdiff": prefill_logits_diff,
        "prefill_store_maxdiff": store_diff,
        "decode_logits_maxdiff": max(decode_diffs),
        "sinkhorn_max_error": sink_err,
        "stream_ptp": stream_ptp,
    }
    print(json.dumps(report["configs"][name], indent=2), flush=True)
    del model, ref, nnx_grad
    jax.clear_caches()

with open(os.environ["LOCED_PARITY_REPORT"], "w") as f:
    json.dump(report, f, indent=2)
print("\nparity report written")
'''
run_script(
    "arch_parity.py", parity, timeout=7200,
    extra_env={
        "LOCED_PARITY_TOL": "1e-5",
        "LOCED_PARITY_REPORT": str(WORK / "arch_parity_report.json"),
    },
)

## 6. MaxText Muon one-step with mHC + MagicNorm

The default MaxText integration suite deliberately runs the old single-stream
model. This cell builds the tiny model with:

```text
loced_magic_norm=true
loced_mhc_streams=4
loced_mhc_dynamic=true
```

and runs two real MaxText Muon updates. It verifies that the new parameters are
visible to MaxText, sharded, differentiated, and updated.

In [ ]:
mhc_train = r'''
import json
import os
import time

import jax
import numpy as np
from flax import nnx
from maxtext.configs import pyconfig

from loced.maxtext.config import LoCEDMaxTextConfig, loced_config_from_maxtext
from loced.maxtext.train import (
    create_model_and_mesh, create_optimizer, synthetic_batches, train_step,
)

accelerator = os.environ["LOCED_ACCELERATOR"]
n = len(jax.devices())
if accelerator == "gpu":
    mesh_overrides = [f"ici_fsdp_parallelism={n}", "ici_data_parallelism=1"]
else:
    mesh_overrides = ["ici_fsdp_parallelism=1", f"ici_data_parallelism={n}"]

config = pyconfig.initialize(
    [
        "loced-mhc-train", "loced/maxtext/configs/loced_tiny.yml",
        *mesh_overrides,
        "steps=3",
        "per_device_batch_size=2",
        "enable_checkpointing=false",
        "log_config=false",
        "warmup_steps_fraction=0.0",
        "learning_rate_final_fraction=1.0",
        "loced_magic_norm=true",
        "loced_mhc_streams=4",
        "loced_mhc_dynamic=true",
        "skip_jax_distributed_system=true",
    ],
    config_class=LoCEDMaxTextConfig,
)
cfg = loced_config_from_maxtext(config)
expected = (
    549456                         # tiny single-stream parameter count
    + cfg.num_layers * cfg.hidden_size  # MagicNorm
    + cfg.num_visits * (4 * 4 + 4 + 4)  # mhc_res + mhc_pre + mhc_post
    + cfg.num_visits * cfg.hidden_size * (4 * 4 + 2 * 4)  # mhc_dyn
)
model, mesh = create_model_and_mesh(config)
print("params :", f"{model.count_params():,}", "expected", f"{expected:,}")
assert model.count_params() == expected, (
    f"mHC parameter count {model.count_params()} != {expected}")
assert cfg.magic_norm is True
assert cfg.mhc_streams == 4
assert cfg.mhc_dynamic is True

schedule, optimizer = create_optimizer(config, model, mesh)
batch = 2
tokens = next(iter(synthetic_batches(
    cfg.vocab_size, cfg.max_seq_len, batch, 1, seed=0)))
before = jax.tree.leaves(nnx.to_pure_dict(nnx.state(model)))
losses = []
t0 = time.perf_counter()
for _ in range(2):
    loss, metrics = train_step(model, optimizer, tokens, reuse_p=0.5)
    jax.block_until_ready(loss)
    losses.append(float(loss))
elapsed = time.perf_counter() - t0
after = jax.tree.leaves(nnx.to_pure_dict(nnx.state(model)))

assert np.isfinite(losses).all()
assert int(optimizer.step.get_value()) == 2
assert any(not np.array_equal(np.asarray(a), np.asarray(b))
           for a, b in zip(before, after))
print("losses :", losses)
print("step   :", int(optimizer.step.get_value()))
print(f"time   : {elapsed / 2 * 1000:.1f} ms/step (compile included)")

for name in ("mhc_res", "mhc_pre", "mhc_post", "mhc_dyn"):
    param = getattr(model, name)
    print(f"{name:10s}", param.get_value().shape, param.get_value().sharding)

report = {
    "params": int(model.count_params()),
    "expected_params": int(expected),
    "losses": losses,
    "optimizer_step": int(optimizer.step.get_value()),
    "magic_norm": bool(cfg.magic_norm),
    "mhc_streams": int(cfg.mhc_streams),
    "mhc_dynamic": bool(cfg.mhc_dynamic),
    "ms_per_step_compile_included": elapsed / 2 * 1000,
}
with open(os.environ["LOCED_MHC_TRAIN_REPORT"], "w") as f:
    json.dump(report, f, indent=2)
'''
run_script(
    "mhc_train.py", mhc_train, timeout=7200,
    extra_env={
        "LOCED_ACCELERATOR": ACCELERATOR,
        "LOCED_MHC_TRAIN_REPORT": str(WORK / "maxtext_mhc_train_report.json"),
    },
)

## 7. Tiny throughput / memory comparison

This is a plumbing benchmark, not a model-scale throughput claim. It compares
the default tiny model with the same model using MagicNorm + four-stream mHC +
dynamic coefficients.

- global synthetic batch: `per_device_batch_size=4`;
- 2 warmup steps, 10 timed steps;
- reports ms/step, tokens/sec, parameter count, and TPU/GPU peak memory.

In [ ]:
bench = r'''
import gc
import json
import os
import time

import jax
import numpy as np
from maxtext.configs import pyconfig

from loced.maxtext.config import LoCEDMaxTextConfig, loced_config_from_maxtext
from loced.maxtext.train import (
    create_model_and_mesh, create_optimizer, synthetic_batches, train_step,
)

accelerator = os.environ["LOCED_ACCELERATOR"]
n = len(jax.devices())
if accelerator == "gpu":
    mesh_overrides = [f"ici_fsdp_parallelism={n}", "ici_data_parallelism=1"]
else:
    mesh_overrides = ["ici_fsdp_parallelism=1", f"ici_data_parallelism={n}"]

warmup = int(os.environ.get("LOCED_BENCH_WARMUP", "2"))
steps = int(os.environ.get("LOCED_BENCH_STEPS", "10"))
batch = int(os.environ.get("LOCED_BENCH_BATCH", "4"))


def make(overrides):
    config = pyconfig.initialize(
        [
            "loced-bench", "loced/maxtext/configs/loced_tiny.yml",
            *mesh_overrides,
            f"steps={warmup + steps}",
            f"per_device_batch_size={batch}",
            "enable_checkpointing=false",
            "log_config=false",
            "skip_jax_distributed_system=true",
            *overrides,
        ],
        config_class=LoCEDMaxTextConfig,
    )
    cfg = loced_config_from_maxtext(config)
    model, mesh = create_model_and_mesh(config)
    _, optimizer = create_optimizer(config, model, mesh)
    return cfg, model, optimizer


def bench(label, overrides):
    cfg, model, optimizer = make(overrides)
    batches = list(synthetic_batches(
        cfg.vocab_size, cfg.max_seq_len, batch, warmup + steps, seed=1))
    for i in range(warmup):
        loss, _ = train_step(model, optimizer, batches[i], reuse_p=0.5)
        jax.block_until_ready(loss)
    losses = []
    t0 = time.perf_counter()
    for i in range(warmup, warmup + steps):
        loss, _ = train_step(model, optimizer, batches[i], reuse_p=0.5)
        losses.append(float(loss))
        jax.block_until_ready(loss)
    elapsed = time.perf_counter() - t0
    stats = jax.devices()[0].memory_stats() or {}
    out = {
        "label": label,
        "params": int(model.count_params()),
        "batch": batch,
        "seq_len": int(cfg.max_seq_len),
        "warmup_steps": warmup,
        "timed_steps": steps,
        "loss_first": losses[0],
        "loss_last": losses[-1],
        "ms_per_step": elapsed / steps * 1000,
        "tokens_per_sec": steps * batch * cfg.max_seq_len / elapsed,
        "peak_bytes_in_use": stats.get("peak_bytes_in_use"),
        "bytes_limit": stats.get("bytes_limit"),
        "magic_norm": bool(cfg.magic_norm),
        "mhc_streams": int(cfg.mhc_streams),
        "mhc_dynamic": bool(cfg.mhc_dynamic),
    }
    print(json.dumps(out, indent=2), flush=True)
    del model, optimizer
    gc.collect()
    jax.clear_caches()
    return out


baseline = bench(
    "baseline",
    ["loced_magic_norm=false", "loced_mhc_streams=1", "loced_mhc_dynamic=false"],
)
mhc = bench(
    "magic_mhc_dynamic",
    ["loced_magic_norm=true", "loced_mhc_streams=4", "loced_mhc_dynamic=true"],
)
report = {
    "baseline": baseline,
    "magic_mhc_dynamic": mhc,
    "baseline_over_mhc_speed": (
        baseline["ms_per_step"] / mhc["ms_per_step"]
        if mhc["ms_per_step"] else None
    ),
    "mhc_over_baseline_params": mhc["params"] / baseline["params"],
}
with open(os.environ["LOCED_BENCH_REPORT"], "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))
'''
run_script(
    "benchmark.py", bench, timeout=7200,
    extra_env={
        "LOCED_ACCELERATOR": ACCELERATOR,
        "LOCED_BENCH_REPORT": str(WORK / "bench_report.json"),
    },
)

## 8. Machine-readable summary

Writes `/kaggle/working/loced_architecture_report.json`. Download it from the
Kaggle file browser if you want the raw numbers.

In [ ]:
reports = {}
for name in (
    "reference_test_report.json",
    "maxtext_test_report.json",
    "arch_parity_report.json",
    "maxtext_mhc_train_report.json",
    "bench_report.json",
):
    path = WORK / name
    if path.is_file():
        reports[name] = json.loads(path.read_text())
    else:
        raise FileNotFoundError(f"missing report: {path}")

summary = {
    "accelerator": ACCELERATOR,
    "backend": reports["arch_parity_report.json"]["backend"],
    "devices": reports["arch_parity_report.json"]["devices"],
    "reference_tests": reports["reference_test_report.json"],
    "maxtext_tests": reports["maxtext_test_report.json"],
    "parity_tolerance": reports["arch_parity_report.json"]["tolerance"],
    "parity": reports["arch_parity_report.json"]["configs"],
    "maxtext_mhc_train": reports["maxtext_mhc_train_report.json"],
    "benchmark": reports["bench_report.json"],
}
out = WORK / "loced_architecture_report.json"
out.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nWrote", out)
print("MagicNorm + mHC validation: ALL CHECKS PASSED")